# 03 — Cleaning the well-test labels

The 85 well tests are the **labels** every VFM model learns from. One wrong label out of 85 hurts far more
than one wrong minute out of 3.9 million, so we check every row.

**Idea:** the file contains both measured rates and *derived* columns. Physics/definitions say they must agree:

| Check | Formula | Units |
|---|---|---|
| Liquid | `liquid = oil + water` | STB/d |
| Water cut | `WC = 100 · water / (oil + water)` | % |
| GOR (gas-oil ratio) | `GOR = 1000 · gas / oil` | scf/STB (gas in Mscf/d × 1000 = scf/d) |
| Total gas | `total = formation gas + lift gas` | Mscf/d |
| GLR (gas-liquid ratio) | `GLR = 1000 · total gas / liquid` | scf/STB |

If one check fails, the *other* columns tell us which number is wrong and what its true value is.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('..', 'src').resolve()))
import clean
import load

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

raw = load.load_well_tests()
residuals = clean.check_well_tests(raw)
bad = (residuals.abs() > 2).any(axis=1)
print(f'{bad.sum()} of {len(raw)} tests fail a consistency check')
pd.concat([raw.loc[bad, ['well_name', 'test_start', 'oil_rate_stbd', 'water_rate_stbd', 'gas_rate_mscfd',
                         'total_gas_rate_mscfd', 'gas_lift_rate_mscfd', 'gor_scf_stb']],
           residuals[bad].round(1).add_prefix('res_')], axis=1)

## Error 1 — sign flips (5 rows)
Rates can't be negative. In each case the other columns give the positive value, e.g.
GL-06 18 Aug: water = −572.2 but liquid 1316.3 = oil 744.1 + **572.2**.

## Error 2 — GL-06 gas recorded in Sm³/d instead of Mscf/d (9 rows)
GL-06 gas is ~28× larger than `oil × GOR / 1000`. 28.317 is exactly the number of m³ in one Mscf
(1 Mscf = 1000 ft³ = 28.317 m³). Total gas has the same problem. GOR and GLR in the file are correct,
which is how we know the rate column is the wrong one.

In [ ]:
ratio = raw.gas_rate_mscfd / (raw.oil_rate_stbd * raw.gor_scf_stb / 1000)
print('gas / (oil x GOR / 1000) per well:')
print(ratio.groupby(raw.well_name).mean().round(2))

## Apply the fixes and re-check

In [ ]:
tests, report = clean.clean_well_tests(raw)
for step, result in report.items():
    print(f'{step}:')
    print('   ', result)

## Cross-check: test sensor averages vs our cleaned minute data
Each test row also stores the average sensor values during the test. If they equal the mean of our
**cleaned** minute data over the same window, (a) the tests were computed from the minute data and
(b) our cleaning did not distort the signal.

In [ ]:
hf = load.load_high_freq_clean()
rows = []
for _, test in tests.iterrows():
    window = hf[(hf.well_name == test.well_name) &
                (hf.timestamp >= test.test_start) & (hf.timestamp < test.test_end)]
    diff = {col: test[col] - window[col].mean() for col in clean.SENSORS}
    diff['shut_in_minutes'] = window.shut_in.sum()
    rows.append(diff)

diff = pd.DataFrame(rows)
print('max |test average - minute mean| per sensor (after fixing labels):')
print(diff[clean.SENSORS].abs().max().round(2))
print('tests that overlap a shut-in:', (diff.shut_in_minutes > 0).sum())

## Trend: oil declines over time (reservoir depletion) → time / cumulative production is a useful feature

In [ ]:
tests.pivot_table(index=tests.test_start.dt.month, columns='well_name',
                  values='oil_rate_stbd', aggfunc='first').round(0)